# Ridge — przewidywanie wartości liczbowej

**Blok III · notebook do samodzielnego uzupełnienia**

Przewiduj wskaźnik progresji choroby na podstawie 10 cech. Porównaj Ridge z naiwną prognozą średniej. Dane mają charakter dydaktyczny, nie kliniczny.

**Zasady:** `random_state=42`, część testowa służy do końcowej oceny; przekształcenia ucz na treningu. W każdej komórce zadania zastąp `raise NotImplementedError(...)` własnym kodem.

### 0. Przygotowanie danych

Ta komórka jest kompletna. Uruchom ją przed rozwiązywaniem zadań.

In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.dummy import DummyRegressor
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

data = load_diabetes(as_frame=True)
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print('Wymiary:', X_train.shape, X_test.shape)
display(X_train.head())

Wymiary: (353, 10) (89, 10)


,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
17,0.070769,0.050680,0.012117,0.056301,0.034206,0.049416,-0.039719,0.034309,0.027364,-0.001078
66,-0.009147,0.050680,-0.018062,-0.033213,-0.020832,0.012152,-0.072854,0.071210,0.000272,0.019633
137,0.005383,-0.044642,0.049840,0.097615,-0.015328,-0.016345,-0.006584,-0.002592,0.017036,-0.013504
245,-0.027310,-0.044642,-0.035307,-0.029770,-0.056607,-0.058620,0.030232,-0.039493,-0.049872,-0.129483
31,-0.023677,-0.044642,-0.065486,-0.081413,-0.038720,-0.053610,0.059685,-0.076395,-0.037129,-0.042499


### Zadanie 1. Punkt odniesienia

Dopasuj `DummyRegressor(strategy="mean")` do treningu. Użyj 5-krotnej walidacji na treningu i zapisz średni dodatni MAE w `mae_baseline_cv`.

In [3]:
from sklearn.model_selection import KFold
cv = KFold(n_splits=5, shuffle=True, random_state=42)

baseline = DummyRegressor(strategy="mean")

baseline_scors  = cross_val_score(baseline, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
mae_baseline_cv = -baseline_scors.mean()

print(f'MAE na CV: {mae_baseline_cv:.3f}')
print(f'Odchylenie standardowe: {baseline_scors.std():.3f}')
print(f'Średni MAE: {baseline_scors.mean():.3f}')


MAE na CV: 66.867
Odchylenie standardowe: 5.632
Średni MAE: -66.867


**Do omówienia:** Dlaczego porównanie z naiwną metodą jest potrzebne?

### Zadanie 2. Model Ridge

Zbuduj `make_pipeline(StandardScaler(), Ridge(alpha=1.0))`. Oceń średnie MAE w 5-krotnym CV na treningu. Skalowanie musi być wewnątrz Pipeline.

In [4]:
model = make_pipeline(StandardScaler(), Ridge(alpha=1.0))

scores = -cross_val_score(model, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
mae_ridge_cv = scores.mean()

print(f'MAE na CV: {mae_ridge_cv:.3f}')
print(f'Odchylenie standardowe: {scores.std():.3f}')
print(f'Średni MAE: {scores.mean():.3f}')

MAE na CV: 44.999
Odchylenie standardowe: 1.808
Średni MAE: 44.999


**Do omówienia:** Czy Ridge poprawia średnie MAE względem baseline?

### Zadanie 3. Siła regularyzacji

Porównaj `alpha` z listy `[0.01, 0.1, 1, 10, 100]` za pomocą 5-krotnej walidacji krzyżowej **tylko na treningu**. Użyj `scoring="neg_mean_absolute_error"` i Pipeline. Pokaż tabelę z dodatnim MAE oraz wybierz najlepszą wartość.

In [8]:
alphas = [0.01, 0.1,1,10,100]
rows = []

for alpha in alphas:
    candidate = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    scores = -cross_val_score(candidate, X_train, y_train, cv=cv, scoring="neg_mean_absolute_error")
    rows.append({
        "alpha":alpha,
        "MAE_CV":scores.mean(),
        "std":scores.std()
    })

wyniki_cv = pd.DataFrame(rows).sort_values("MAE_CV")
best_alpha = wyniki_cv.iloc[0]["alpha"]
display(wyniki_cv)

print(f"alpha: {best_alpha}")


,alpha,MAE_CV,std
2,1.00,44.998879,1.807942
1,0.10,45.041133,1.776535
0,0.01,45.047466,1.772948
3,10.00,45.060435,1.951197
4,100.00,46.047403,2.868708


alpha: 1.0


**Do omówienia:** Dlaczego `neg_mean_absolute_error` zwraca wartości ujemne?

### Zadanie 4. Ostateczny wynik

Dopasuj Ridge z wybranym `alpha` na całym treningu i **raz** oceń na teście. Oblicz MAE, RMSE i R². Porównaj do baseline dopasowanego do całego treningu, ocenionego na tym samym teście. Napisz 2 zdania wniosku.

In [9]:
best_model = make_pipeline(StandardScaler(), Ridge(alpha=best_alpha))
best_model.fit(X_train, y_train)
y_pred = best_model.predict(X_test)

final_mae = mean_absolute_error(y_test, y_pred)
final_rmse = root_mean_squared_error(y_test, y_pred)
final_r2 = r2_score(y_test, y_pred)

baseline_final = DummyRegressor(strategy="mean")
baseline_final.fit(X_train, y_train)
basekline_pred = baseline_final.predict(X_test)

baseline_mae = mean_absolute_error(y_test, basekline_pred)

print(f"wybrana alpha: {best_alpha}")
print(f"MAE na zbiorze testowym: {final_mae:.3f}")
print(f"RMSE na zbiorze testowym: {final_rmse:.3f}")
print(f"R² na zbiorze testowym: {final_r2:.3f}")
print(f"MAE na zbiorze testowym baseline: {baseline_mae:.3f}")

wybrana alpha: 1.0
MAE na zbiorze testowym: 42.812
RMSE na zbiorze testowym: 53.777
R² na zbiorze testowym: 0.454
MAE na zbiorze testowym baseline: 64.006


**Do omówienia:** Co oznacza R² ujemne albo bliskie zeru? Czy poprawa na CV gwarantuje poprawę na teście?

---
**Dokumentacja:** https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html

**Oddanie:** zapisz działający plik `.ipynb` z wynikami i krótkimi odpowiedziami pod każdym zadaniem.